# MeAJOR Classical Baselines

This notebook evaluates classical machine learning baselines on the processed MeAJOR dataset 
using TF-IDF features and a fixed 60/40 train-test split.

Key methodological choices:
- `class_weight='balanced'` is applied to Logistic Regression and Linear SVC.
- `class_weight='balanced_subsample'` is applied to Random Forest.
- `GridSearchCV` is used for Logistic Regression and Linear SVC to tune hyperparameters on training data only.

## 1. Import Libraries and Load Processed Data

In [ ]:
import time
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
train_path = Path('../data/processed/meajor/meajor_train_60.parquet')
test_path  = Path('../data/processed/meajor/meajor_test_40.parquet')

train_df = pd.read_parquet(train_path)
test_df  = pd.read_parquet(test_path)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

## 2. Verify the Saved Split

In [ ]:
print('Train columns:', train_df.columns.tolist())
print('Test columns:', test_df.columns.tolist())
print()
print('Train label distribution:')
print(train_df['label'].value_counts())
print()
print(train_df['label'].value_counts(normalize=True))
print()
print('Test label distribution:')
print(test_df['label'].value_counts())
print()
print(test_df['label'].value_counts(normalize=True))

In [ ]:
display(train_df.head(3))

## 3. TF-IDF Vectorisation

The vectoriser is fitted on training data only (fit_transform) and applied to the test set 
(transform only) to prevent data leakage.

In [ ]:
X_train = train_df['text']
y_train = train_df['label']
X_test  = test_df['text']
y_test  = test_df['label']

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000
)

# Fit ONLY on training data, then transform both splits
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print('TF-IDF train shape:', X_train_tfidf.shape)
print('TF-IDF test shape:', X_test_tfidf.shape)
print('Vocabulary size:', X_train_tfidf.shape[1])

## 4. Define and Tune Models

- **MultinomialNB**: No class_weight support; no tuning needed.
- **Logistic Regression**: `class_weight='balanced'`; C tuned via GridSearchCV.
- **Linear SVC**: `class_weight='balanced'`; C tuned via GridSearchCV.
- **Random Forest**: `class_weight='balanced_subsample'`.

In [ ]:
# --- Multinomial NB ---
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# --- Logistic Regression with GridSearchCV ---
lr_base = LogisticRegression(
    max_iter=1000, solver='liblinear', random_state=42,
    class_weight='balanced'
)
lr_grid = GridSearchCV(lr_base, param_grid={'C': [0.01, 0.1, 1, 10]}, cv=5, scoring='f1', n_jobs=-1)
lr_grid.fit(X_train_tfidf, y_train)
print(f'Best LR params: {lr_grid.best_params_}')

# --- Linear SVC with GridSearchCV ---
svc_base = LinearSVC(random_state=42, class_weight='balanced')
svc_grid = GridSearchCV(svc_base, param_grid={'C': [0.01, 0.1, 1, 10]}, cv=5, scoring='f1', n_jobs=-1)
svc_grid.fit(X_train_tfidf, y_train)
print(f'Best SVC params: {svc_grid.best_params_}')

# --- Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced_subsample', n_jobs=-1)
rf_model.fit(X_train_tfidf, y_train)

## 5. Evaluate All Models

Metrics: Accuracy, Precision, Recall, F1, False Positive Rate, Inference Time.

In [ ]:
figures_dir = Path('../results/figures/')
figures_dir.mkdir(parents=True, exist_ok=True)

models = {
    'TF-IDF + MultinomialNB':      nb_model,
    'TF-IDF + LogisticRegression': lr_grid.best_estimator_,
    'TF-IDF + LinearSVM':          svc_grid.best_estimator_,
    'TF-IDF + RandomForest':       rf_model,
}

all_results = []

for name, model in models.items():
    print(f"\n{'='*55}")
    print(f'  {name}')
    print(f"{'='*55}")

    start = time.perf_counter()
    y_pred = model.predict(X_test_tfidf)
    elapsed = time.perf_counter() - start

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    print(classification_report(y_test, y_pred, target_names=['Legitimate (0)', 'Phishing (1)']))
    print(f'  False Positive Rate : {fpr:.4f}')
    print(f'  Inference Time      : {elapsed / len(y_test) * 1000:.4f} ms/email')

    all_results.append({
        'Dataset':   'MeAJOR',
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_test, y_pred, zero_division=0), 4),
        'FPR':       round(fpr, 4),
        'Inference_ms_per_email': round(elapsed / len(y_test) * 1000, 6),
        'train_rows':  len(train_df),
        'test_rows':   len(test_df),
        'vocab_size':  X_train_tfidf.shape[1],
    })

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Phishing'])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix — {name}')
    plt.tight_layout()
    safe_name = name.lower().replace(' ', '_').replace('+', '').replace('__', '_')
    plt.savefig(figures_dir / f'meajor_cm_{safe_name}.png', dpi=150)
    plt.show()

## 6. Save Results

In [ ]:
results_classical = pd.DataFrame(all_results)
display(results_classical)

results_dir = Path('../results/metrics')
results_dir.mkdir(parents=True, exist_ok=True)
results_classical.to_csv(results_dir / 'meajor_classical_baselines_results.csv', index=False)
print('Saved to:', results_dir / 'meajor_classical_baselines_results.csv')